# Infer Notebook

- Source: `src/infer.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""단일 이미지 추론(Inference).

이 스크립트가 하는 일
─────────────────
1. 이미지 1장을 읽어 validation과 동일한 전처리를 적용
2. checkpoint 로드 후 모델을 eval() 모드로 실행
3. softmax 확률 + 예측 클래스를 되돌려줘줌
4. Streamlit 앱(app/streamlit_app.py)에서도 이 predict() 함수를 재사용한다.

사용 예시
─────────
    python -m src.infer --image data/raw/defect/full_black/001.jpg --config configs/defect.yaml

반환값 예시
─────────
    {"label": "full_black", "prob": 0.97, "all_probs": {"broken": 0.01, ...}}
"""
from __future__ import annotations
import argparse
from pathlib import Path
import numpy as np
import torch
from PIL import Image

from src.utils.config import load_config
from src.dataset import get_transforms
from src.model import CoffeeClassifier
from src.evaluate import load_checkpoint


## Step 2. Function: predict

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def predict(image_path: str, cfg: dict, ckpt_path: str | None = None,
            device: str | None = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    task, classes = cfg["task"], cfg["classes"]
    c2i = {c: i for i, c in enumerate(classes)}
    idx_to_class = {i: c for c, i in c2i.items()}

    ckpt_path = ckpt_path or f"{cfg['paths']['ckpt_dir']}/best_{task}.pth"
    _, state = load_checkpoint(ckpt_path, device)

    # train/evaluate와 같은 구조의 모델을 다시 만들고 weight를 로드한다.
    model = CoffeeClassifier(
        backbone=cfg["model"]["name"], n_classes=len(classes), pretrained=False,
        dropout=cfg["model"]["dropout"], hidden=cfg["model"]["hidden"],
    ).to(device)
    model.load_state_dict(state); model.eval()

    # 단일 이미지라도 학습 때의 validation transform과 같은 전처리를 적용한다.
    tf = get_transforms(task=task, train=False, img_size=cfg["data"]["img_size"])
    img = np.array(Image.open(image_path).convert("RGB"))
    x = tf(image=img)["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        # 모델 출력은 logits이므로 사람이 읽기 쉬운 확률로 바꾼다.
        probs = torch.softmax(model(x), dim=1).cpu().numpy()[0]
    top = int(probs.argmax())
    return {
        "label": idx_to_class[top],
        "prob": float(probs[top]),
        "all_probs": {idx_to_class[i]: float(probs[i]) for i in range(len(classes))},
    }


## Step 3. Function: main

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--ckpt", default=None)
    args = ap.parse_args()
    cfg = load_config(args.config)
    out = predict(args.image, cfg, args.ckpt)
    print(out)


## Step 4. Run Entry Point

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
if __name__ == "__main__":
    main()


## 실행 파라미터 가이드

- 이 파일은 원래 CLI 인자(argparse) 기반으로 동작합니다.
- 노트북에서는 인자 대신 아래처럼 변수 셀을 만들어 실행하세요.


In [ ]:
# 예시 파라미터 셀
CONFIG_PATH = 'configs/default.yaml'
CKPT_PATH = None
